# Germana: stopwords y tres configuraciones de BERTopic

Se reutilizan los mismos 1,177 párrafos y embeddings de MiniLM del notebook 01, en el mismo orden. Se conserva el texto completo para los embeddings. `CountVectorizer` elimina la lista española de NLTK al generar las palabras descriptivas de los tópicos. Se guarda la lista exacta utilizada.

| Configuración | min_cluster_size | min_samples | ngram_range |
|---|---:|---:|---|
| A: grupos específicos | 15 | 5 | (1, 1) |
| B: agrupamiento más conservador | 30 | 10 | (1, 1) |
| C: B con bigramas | 30 | 10 | (1, 2) |

A frente a B evalúa un cambio conjunto en los parámetros de densidad: no permite atribuir un efecto exclusivamente a uno de ellos. Se espera que B favorezca grupos mayores y sea más exigente al asignar documentos, pero se comprobará con los resultados. B frente a C aísla el efecto de los bigramas en la representación; los grupos y outliers deben permanecer idénticos. C se construye actualizando la representación de una copia de B, sin repetir el agrupamiento.

UMAP fijo: 15 vecinos, 5 componentes, min_dist=0, métrica coseno, random_state=42, n_jobs=1. HDBSCAN: distancia euclídea en el espacio reducido, cluster_selection_method="eom". No se fuerza un número de tópicos (`nr_topics=None`), ni se reasignan outliers durante esta comparación. CountVectorizer usa min_df=1, minúsculas y conserva acentos; no eliminamos nombres de personajes ni títulos nobiliarios por adelantado.


In [1]:
from pathlib import Path
from copy import deepcopy
import hashlib
import importlib.metadata
import json
import time

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

BASE = Path.cwd() if (Path.cwd() / "germana.txt").exists() else Path.cwd() / "BERTopic"
OUT = BASE / "resultados" / "configuraciones"
OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(BASE / "resultados" / "segmentos.csv")
embeddings = np.load(BASE / "resultados" / "embeddings_minilm.npy")
docs = df.texto.tolist()
assert embeddings.shape == (len(docs), 384)
assert np.isfinite(embeddings).all()

nltk_dir = BASE / ".cache" / "nltk_data"
nltk.data.path.insert(0, str(nltk_dir))
stop_file = OUT / "stopwords_es.txt"
if stop_file.exists():
    stop_es = stop_file.read_text().splitlines()
else:
    try:
        stop_es = sorted(set(stopwords.words("spanish")))
    except LookupError:
        nltk.download("stopwords", download_dir=str(nltk_dir), raise_on_error=True)
        stop_es = sorted(set(stopwords.words("spanish")))
    stop_file.write_text("\n".join(stop_es) + "\n")
print(f"Documentos: {len(docs)}; stopwords españolas: {len(stop_es)}")
print("Ejemplos:", ", ".join(stop_es[:20]))


[directorio del proyecto]/BERTopic/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[directorio del proyecto]/BERTopic/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Documentos: 1177; stopwords españolas: 313
Ejemplos: a, al, algo, algunas, algunos, ante, antes, como, con, contra, cual, cuando, de, del, desde, donde, durante, e, el, ella


In [2]:
CONFIGS = {
    "A": {"min_cluster_size": 15, "min_samples": 5, "ngram_range": (1, 1)},
    "B": {"min_cluster_size": 30, "min_samples": 10, "ngram_range": (1, 1)},
    "C": {"min_cluster_size": 30, "min_samples": 10, "ngram_range": (1, 2)},
}

def vectorizer(ngram_range):
    return CountVectorizer(stop_words=stop_es, ngram_range=ngram_range,
                           min_df=1, lowercase=True, strip_accents=None)

models, elapsed = {}, {}
for name in ["A", "B"]:
    config = CONFIGS[name]
    started = time.perf_counter()
    model = BERTopic(
        language="multilingual", embedding_model=None,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                        metric="cosine", random_state=42, n_jobs=1),
        hdbscan_model=HDBSCAN(min_cluster_size=config["min_cluster_size"],
                             min_samples=config["min_samples"], metric="euclidean",
                             cluster_selection_method="eom", prediction_data=True),
        vectorizer_model=vectorizer(config["ngram_range"]),
        nr_topics=None, top_n_words=10, calculate_probabilities=False,
        verbose=False,
    )
    model.fit_transform(docs, embeddings=embeddings)
    models[name] = model
    elapsed[name] = time.perf_counter() - started
    print(f"{name}: ajuste completado en {elapsed[name]:.2f} s", flush=True)

started = time.perf_counter()
models["C"] = deepcopy(models["B"])
models["C"].update_topics(docs, vectorizer_model=vectorizer((1, 2)), top_n_words=10)
elapsed["C"] = time.perf_counter() - started
assert np.array_equal(models["B"].topics_, models["C"].topics_)
print(f"C: actualización de representación completada en {elapsed['C']:.2f} s")


A: ajuste completado en 4.55 s


B: ajuste completado en 1.83 s


C: actualización de representación completada en 0.08 s


In [3]:
from plotly.offline import get_plotlyjs
(OUT / "plotly.min.js").write_text(get_plotlyjs(), encoding="utf-8")
summaries = []
assignments = df[["segmento_id", "capitulo", "texto"]].copy()
for name, model in models.items():
    folder = OUT / name
    folder.mkdir(exist_ok=True)
    info = model.get_topic_info()
    info.to_csv(folder / "topic_info.csv", index=False)
    topics = np.asarray(model.topics_)
    assignments[f"topico_{name}"] = topics
    valid_ids = sorted(set(topics) - {-1})
    rep_rows = []
    for topic in valid_ids:
        # En C se conservan los ejemplos de B para comparar las mismas agrupaciones.
        texts = model.get_representative_docs(int(topic)) or []
        for rank, text in enumerate(texts[:3], 1):
            matching = df.loc[(df.texto == text) & (topics == topic)]
            rep_rows.append({"topico": int(topic), "orden": rank,
                "segmento_id": matching.iloc[0].segmento_id if len(matching) else "",
                "texto": text})
    pd.DataFrame(rep_rows).to_csv(folder / "ejemplos_representativos.csv", index=False)
    word_lists = [[word for word, score in model.get_topic(int(t))[:10]
                   if word and score > 0] for t in valid_ids]
    all_words = [word for words in word_lists for word in words]
    sizes = info.loc[info.Topic != -1, "Count"]
    summaries.append({"configuracion": name, **CONFIGS[name],
        "topicos_sin_outliers": len(valid_ids), "outliers": int((topics == -1).sum()),
        "porcentaje_outliers": float(100 * (topics == -1).mean()),
        "menor_topico": int(sizes.min()) if len(sizes) else 0,
        "mayor_topico": int(sizes.max()) if len(sizes) else 0,
        "diversidad_top10": len(set(all_words)) / len(all_words) if all_words else None,
        "segundos_operacion": elapsed[name],
        "operacion": "actualizar palabras de B" if name == "C" else "ajustar modelo"})
    if valid_ids:
        model.visualize_barchart(top_n_topics=min(12, len(valid_ids)), n_words=8,
            title=f"Configuración {name}: palabras por tópico").write_html(
                str(folder / "barchart.html"), include_plotlyjs="../plotly.min.js")
    if len(valid_ids) >= 2:
        model.visualize_hierarchy().write_html(str(folder / "jerarquia.html"), include_plotlyjs="../plotly.min.js")
    print(f"\n{name}: tabla de tópicos")
    print(info[["Topic", "Count", "Name"]].to_string(index=False))

comparison = pd.DataFrame(summaries)
comparison.to_csv(OUT / "comparacion.csv", index=False)
assignments.to_csv(OUT / "asignaciones.csv", index=False)
metadata = {
    "configuraciones": CONFIGS,
    "umap": {"n_neighbors": 15, "n_components": 5, "min_dist": 0.0,
             "metric": "cosine", "random_state": 42, "n_jobs": 1},
    "stopwords_fuente": "NLTK spanish; lista exacta en stopwords_es.txt",
    "stopwords_sha256": hashlib.sha256(stop_file.read_bytes()).hexdigest(),
    "embeddings_sha256": hashlib.sha256((BASE / "resultados" / "embeddings_minilm.npy").read_bytes()).hexdigest(),
    "segmentos_sha256": hashlib.sha256((BASE / "resultados" / "segmentos.csv").read_bytes()).hexdigest(),
    "ARI_A_B_incluye_outliers": adjusted_rand_score(models["A"].topics_, models["B"].topics_),
    "ARI_B_C_incluye_outliers": adjusted_rand_score(models["B"].topics_, models["C"].topics_),
    "versiones": {p: importlib.metadata.version(p) for p in
        ["bertopic", "hdbscan", "umap-learn", "scikit-learn", "nltk", "numpy", "plotly"]}}
(OUT / "metadatos.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
print("\nComparación:")
print(comparison.to_string(index=False))



A: tabla de tópicos
 Topic  Count                              Name
    -1    235    -1_señora_si_germana_villanera
     0    331            0_usted_si_bien_señora
     1     83      1_chermidy_señora_mujer_hijo
     2     76         2_duquesa_duque_casa_años
     3     65       3_duque_señor_usted_duquesa
     4     64       4_bris_doctor_señor_germana
     5     51        5_doctor_usted_dijo_médico
     6     44        6_francos_mil_dinero_usted
     7     41          7_cama_balcón_paso_duque
     8     35    8_germana_corazón_alegría_vida
     9     31         9_jardín_cabeza_casa_gran
    10     22   10_dijo_levantó_lágrimas_oculta
    11     21     11_parís_usted_pase_prohibida
    12     21 12_veneno_venenos_pulmón_arsénico
    13     21        13_ly_gu_misioneros_tcheou
    14     21 14_chermidy_señora_doctor_mantoux
    15     15          15_mar_ciudad_noche_allí

B: tabla de tópicos
 Topic  Count                          Name
    -1    251      -1_usted_señora_si_señor
     0

## Cómo interpretar esta comparación

- Menos outliers no significa automáticamente mejor modelo: revisar que los párrafos asignados compartan un tema reconocible.
- La diversidad top-10 es la fracción de términos distintos entre las palabras principales. Es descriptiva; no mide coherencia semántica y los bigramas pueden aumentarla sin mejorar los tópicos.
- El tiempo de C corresponde solamente a actualizar las palabras y no es comparable con el ajuste completo de A o B. El primer ajuste también puede incluir compilación de UMAP.
- B y C conservan los mismos grupos. Se debe revisar si los bigramas ayudan a nombrarlos e interpretarlos, sin afirmar que mejoran el agrupamiento.
- Los nombres de `get_topic_info()` son automáticos. Todavía falta asignar nombres propios, interpretar las visualizaciones y discutir los ejemplos.
- Outliers: se mantienen como -1 y se reportan sin reasignación, para conservar una comparación inicial transparente.

Fuentes:
- https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html
- https://maartengr.github.io/BERTopic/getting_started/vectorizers/vectorizers.html
- https://maartengr.github.io/BERTopic/faq.html
- https://www.nltk.org/howto/corpus.html

Uso de IA: Codex (OpenAI), 25 de octubre de 2026, para proponer y programar las configuraciones y guardar los resultados. La interpretación y las decisiones deben revisarse y explicarse en el reporte.
